# 🔬 Notebook 3: Netflix — Deep Dive: Encoding, ABR, Recs, CDN

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — the encoding pipeline

When a new mezzanine (master) file arrives, we need to produce many renditions.

```
  upload -> S3  -->  encoding queue (SQS/Kafka)
                        |
                  +-----+------+
                  v            v
             ffmpeg worker   ffmpeg worker        (N workers)
                  |            |
                  v            v
            240p.ts       1080p.ts   -> S3 -> CDN
                  |
                  v
         manifest builder -> catalog DB
```

Key properties:
- **Fan-out per rendition**. A 2-hour movie x 6 renditions = 6 independent jobs.
- **Chunked encoding**: split the mezzanine into e.g. 3-minute segments. 40 workers per
  movie -> done in minutes, not hours.
- **Idempotent jobs** keyed by `(asset_id, rendition, chunk)`: retrying after a crash is safe.

Let's compare three implementations.

In [ ]:
import time, random
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

@dataclass(frozen=True)
class EncodeJob:
    asset_id: int
    rendition: str
    chunk_idx: int = 0   # for chunked encoding

RENDITIONS = ("240p","360p","480p","720p","1080p","4k")
CHUNKS_PER_MOVIE = 10  # imagine a 2-hour movie split into 10 segments

# Per-rendition time for the WHOLE movie. A chunk costs 1/CHUNKS_PER_MOVIE of that,
# because encoding work is proportional to video duration.
FULL_MOVIE_TIME = {"240p":0.30,"360p":0.36,"480p":0.48,"720p":0.72,"1080p":1.08,"4k":1.80}

def fake_encode(job: EncodeJob, chunked: bool=False) -> str:
    total = FULL_MOVIE_TIME[job.rendition]
    work = total / CHUNKS_PER_MOVIE if chunked else total
    time.sleep(work)
    return f"{job.asset_id}-{job.rendition}-{job.chunk_idx}.ts"


In [ ]:
# --- v1 (BAD): serial, one rendition at a time, whole movie per job ---
random.seed(0)
jobs_v1 = [EncodeJob(42, r) for r in RENDITIONS]
t0 = time.time()
results_v1 = [fake_encode(j, chunked=False) for j in jobs_v1]
print(f"v1 serial whole-movie: {time.time()-t0:.2f}s  -> {len(results_v1)} outputs")


In [ ]:
# --- v2 (BETTER): parallel across renditions (6 workers) ---
random.seed(0)
jobs_v2 = [EncodeJob(42, r) for r in RENDITIONS]
t0 = time.time()
with ThreadPoolExecutor(max_workers=6) as pool:
    results_v2 = list(pool.map(lambda j: fake_encode(j, chunked=False), jobs_v2))
print(f"v2 parallel renditions: {time.time()-t0:.2f}s  -> {len(results_v2)} outputs")


In [ ]:
# --- v3 (BEST): parallel across renditions AND chunks ---
# 6 renditions x 10 chunks = 60 small jobs.
random.seed(0)
jobs_v3 = [EncodeJob(42, r, c) for r in RENDITIONS for c in range(CHUNKS_PER_MOVIE)]
t0 = time.time()
with ThreadPoolExecutor(max_workers=20) as pool:
    results_v3 = list(pool.map(lambda j: fake_encode(j, chunked=True), jobs_v3))
print(f"v3 chunked + parallel: {time.time()-t0:.2f}s  -> {len(results_v3)} outputs")
print("\nNote: v3 produces MORE outputs but finishes FASTEST - that is horizontal scale.")
print("The longest rendition (4k) no longer bottlenecks, because its chunks run in parallel.")


## Deep dive 2 — ABR (Adaptive Bitrate) streaming

The player measures its own bandwidth every chunk and picks the variant that will arrive
*before* the buffer runs dry. Let's simulate a bumpy network.

In [ ]:
VARIANTS = [  # (name, bitrate Mbps)
    ("240p", 0.4), ("360p", 0.8), ("480p", 1.4),
    ("720p", 2.8), ("1080p", 5.0), ("4k", 15.0),
]
CHUNK_SECONDS   = 4      # each chunk covers 4 seconds of video
STARTUP_BUFFER  = 4.0    # player pre-buffers one chunk before playback starts

def simulate(bandwidth_profile_mbps, strategy="abr"):
    """Play a stream over a given bandwidth profile and count rebuffer events.

    IMPORTANT: a real player cannot see the future. It picks the next chunk's quality
    from the throughput it MEASURED on the chunk it just downloaded. Handing the chooser
    the current chunk's bandwidth would give it perfect foresight and make ABR look far
    better than it is — the single most common way this simulation is written wrong.
    """
    buffer_s   = STARTUP_BUFFER
    rebuffers  = 0
    stalled_s  = 0.0
    picked     = []
    estimate   = bandwidth_profile_mbps[0]      # bootstrap: first chunk is a guess

    for measured in bandwidth_profile_mbps:
        if strategy == "abr":
            # decide using the LAGGED estimate, not `measured`
            feasible = [v for v in VARIANTS if v[1] <= 0.8 * estimate]
            chosen = max(feasible, key=lambda v: v[1]) if feasible else VARIANTS[0]
        else:                                    # fixed 1080p — naive client
            chosen = next(v for v in VARIANTS if v[0] == "1080p")
        picked.append(chosen[0])

        # the chunk downloads at the ACTUAL bandwidth for this moment
        download_time = chosen[1] * CHUNK_SECONDS / measured
        buffer_s -= download_time
        if buffer_s < 0:                         # buffer ran dry -> playback stalls
            rebuffers += 1
            stalled_s += -buffer_s
            buffer_s = 0.0
        buffer_s += CHUNK_SECONDS

        estimate = measured                      # what we learned, used for the NEXT chunk
    return picked, rebuffers, stalled_s

# A network that starts great, collapses, then recovers
bw = [8, 8, 8, 1.0, 1.0, 0.6, 0.6, 2.5, 5, 8]

for strat in ("fixed1080p", "abr"):
    picked, rebuf, stalled = simulate(bw, strategy=strat)
    print(f"{strat:11} -> rebuffers={rebuf}  stalled={stalled:5.1f}s")
    print(f"{'':11}    picked={picked}")

print("""
Notice ABR still rebuffers (see the count above) — it just stalls for a fraction of the
time the naive player does. That is the honest result: because the player reacts to
the PREVIOUS chunk, a sudden cliff (8 Mbps -> 1 Mbps) always catches it holding a chunk
that is too big. ABR cannot prevent the first stall after a collapse; it can only stop
the bleeding afterwards. Production players add a buffer-occupancy term (BOLA/BBA) on top
of throughput so that a draining buffer forces a downshift even before throughput drops —
and they still cannot beat physics on the first chunk.""")


ABR keeps the buffer healthy by dropping quality instead of stalling. A player that
insists on 1080p over a 1 Mbps link will rebuffer constantly.

## Deep dive 3 — recommendations

Netflix's recommender is offline-precomputed so serving is just a key lookup:

```
  user_features   item_features
        +------+------+
               v
      +------------------+
      | offline training |   (nightly / weekly)
      |  collaborative   |
      |  filtering + ML  |
      +--------+---------+
               | emit top-K per user
               v
         recs_cache (Redis)
               ^
  GET /recs/for-me -> O(1) lookup
```

We'll implement a **toy item-based collaborative filter** with cosine similarity, then
compare "compute-on-request" vs "precomputed lookup".

In [ ]:
import math, time
from collections import defaultdict

# user -> set of liked titles
ratings = {
    "alice": {"the matrix", "inception", "interstellar"},
    "bob":   {"the matrix", "inception", "memento"},
    "carol": {"interstellar", "arrival"},
    "dave":  {"memento", "inception"},
    "erin":  {"the matrix", "memento", "arrival"},
}

title_users = defaultdict(set)
for user, titles in ratings.items():
    for t in titles:
        title_users[t].add(user)

def cosine(a: set, b: set) -> float:
    if not a or not b: return 0.0
    return len(a & b) / math.sqrt(len(a) * len(b))

def recommend(user: str, k=3):
    watched = ratings[user]
    scores = defaultdict(float)
    for wt in watched:
        for other in title_users:
            if other in watched: continue
            scores[other] += cosine(title_users[wt], title_users[other])
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

for u in ratings:
    print(f"{u:6} -> {recommend(u)}")


In [ ]:
# --- Bad: compute on every request ---
t0 = time.time()
for _ in range(1000):
    _ = recommend("alice")
on_demand_ms = (time.time() - t0) * 1000
print(f"on-demand (1000 reqs): {on_demand_ms:.1f} ms")

# --- Best: precompute once, serve from a dict (our "Redis") ---
recs_cache = {u: recommend(u) for u in ratings}
t0 = time.time()
for _ in range(1000):
    _ = recs_cache["alice"]
cached_ms = (time.time() - t0) * 1000
print(f"precomputed lookup   : {cached_ms:.3f} ms")
print(f"speedup              : {on_demand_ms / max(cached_ms, 1e-6):,.0f}x")


### Why offline precomputation is the right default
- Recommendations don't need to be second-by-second fresh. Nightly is fine.
- Training is CPU-heavy; serving must be O(1).
- Freshness gaps are patched by a lightweight re-ranker at serve time (e.g. deboost
  titles the user just watched).


## Deep dive 4 — CDN math, in numbers

Napkin math:
- 30M concurrent viewers x 3 Mbps = 90 Tbps.
- Typical cloud region egress cap: ~10 Tbps.
- 200 edge POPs x ~450 Gbps each ≈ 90 Tbps. We're in range.

CDN wins on three axes:
1. **Bandwidth**: massive aggregate capacity close to users.
2. **Latency**: 20ms RTT vs 150ms trans-ocean.
3. **Origin protection**: one popular episode launch could DoS the origin; the CDN absorbs
   the spike.

This is why Netflix built **OpenConnect** — caching appliances placed *inside* ISP networks.
The most popular bytes never even touch the public internet backbone.


In [ ]:
# What cache hit rate do we need so origin fits in one region?
total_tbps = 90
region_cap = 10
needed_hit_rate = 1 - region_cap / total_tbps
print(f"Need hit rate >= {needed_hit_rate:.1%} to keep origin under {region_cap} Tbps")

print("\nAnd the cost of being wrong:")
for hr in (0.80, 0.90, 0.95, 0.99):
    origin = total_tbps * (1 - hr)
    status = "OK" if origin <= region_cap else "OVERLOAD"
    print(f"  hit={hr:.0%} -> origin={origin:5.1f} Tbps ({status})")


## Takeaways
1. **Encoding scales by splitting work two ways**: across renditions *and* across chunks.
2. **ABR is about the buffer, not the bitrate.** Players optimize for "don't stall."
3. **Recs are precomputed.** Serving is a dict lookup; training is a batch job.
4. **CDN hit rate is the whole economic story** of a streaming service.
